# 00 — Vector Addition

**Goal:** compare a built-in PyTorch addition with a small Triton kernel.

This is the starting experiment. It teaches how a Triton program handles a group of elements, how masks protect the final incomplete block, and how to run a basic GPU timing test.

Run the cells from top to bottom in Google Colab with a GPU runtime. No performance result is assumed in advance.

## 1. Check the environment

PyTorch supplies the tensors and GPU runtime. Triton lets us write a small GPU kernel in Python-like syntax. If the check says CUDA is unavailable, select **Runtime → Change runtime type → T4 GPU (or another available GPU)** in Colab.

In [1]:
import math
import statistics
import time
import torch
import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Google Colab, then run this notebook again.")

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Triton: 3.6.0
GPU: Tesla T4


## 2. The operation

For each position `i`, calculate `out[i] = x[i] + y[i]`. A mask is needed because the number of elements is not always a perfect multiple of the block size.

In [2]:
@triton.jit
def vector_add_kernel(x_ptr, y_ptr, out_ptr, n_elements: tl.constexpr,
                      BLOCK_SIZE: tl.constexpr):
    program_id = tl.program_id(0)
    offsets = program_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def triton_vector_add(x, y, out, block_size=256):
    grid = (triton.cdiv(x.numel(), block_size),)
    vector_add_kernel[grid](x, y, out, x.numel(), BLOCK_SIZE=block_size)
    return out

### What the kernel lines mean

- `tl.program_id(0)` identifies which block of work this program handles.
- `tl.arange(0, BLOCK_SIZE)` makes positions inside that block.
- `offsets` combines the block number and local positions to make global array positions.
- `mask` is true only for positions that exist in the input arrays.
- `tl.load` reads values, and `tl.store` writes the answer.

Triton programs are not the same thing as individual CUDA threads. Triton describes blocks of work; the compiler maps that work to the GPU.

## 3. Check correctness

We use an odd array length so the last block is only partly full. The reference result comes from ordinary PyTorch addition.

In [3]:
N = 1_000_003
x = torch.randn(N, device="cuda", dtype=torch.float32)
y = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.empty_like(x)

triton_vector_add(x, y, out, block_size=256)
expected = x + y
torch.testing.assert_close(out, expected)
print("Correctness check passed for", N, "elements")

Correctness check passed for 1000003 elements


## 4. Basic timing

A single CPU stopwatch around a GPU call can mostly measure how fast the CPU submits work, because GPU execution is often asynchronous. CUDA events mark positions on the GPU's work timeline. We warm up first, time multiple calls, repeat the measurement, and use the median to reduce the effect of an unusual reading.

This is an introductory benchmark, not a substitute for a profiler. Very small kernels are especially sensitive to launch overhead and runtime conditions.

In [4]:
def median_gpu_time_ms(fn, warmup=10, repeats=50, rounds=5):
    """Measure repeated calls with CUDA events; return median ms per call and all rounds."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    samples = []
    for _ in range(rounds):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats):
            fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) / repeats)

    return statistics.median(samples), samples

In [5]:
N = 1 << 20
x = torch.randn(N, device="cuda", dtype=torch.float32)
y = torch.randn(N, device="cuda", dtype=torch.float32)
out_torch = torch.empty_like(x)
out_triton = torch.empty_like(x)


def run_torch():
    torch.add(x, y, out=out_torch)


def run_triton():
    triton_vector_add(x, y, out_triton, block_size=256)

run_torch()
run_triton()
torch.testing.assert_close(out_torch, out_triton)

torch_ms, torch_samples = median_gpu_time_ms(run_torch)
triton_ms, triton_samples = median_gpu_time_ms(run_triton)
print(f"PyTorch median: {torch_ms:.6f} ms/call")
print(f"Triton median:  {triton_ms:.6f} ms/call")
print(f"PyTorch / Triton time ratio: {torch_ms / triton_ms:.3f}x")

PyTorch median: 0.055092 ms/call
Triton median:  0.053108 ms/call
PyTorch / Triton time ratio: 1.037x


## 5. Estimate effective bandwidth

For each `float32` output, this simple operation reads two 4-byte values and writes one 4-byte value: 12 bytes per element. Dividing this estimated traffic by the elapsed time gives an **effective bandwidth estimate**, not a hardware-counter reading of actual DRAM traffic.

In [6]:
bytes_estimate = 3 * N * 4
seconds = triton_ms / 1000.0
effective_gb_s = bytes_estimate / seconds / 1e9
print(f"Estimated effective bandwidth: {effective_gb_s:.2f} GB/s")

Estimated effective bandwidth: 236.93 GB/s


## Notes to record

Record the GPU name, PyTorch/Triton versions, array length, block size, correctness result, and your actual timing. Do not claim Triton is faster unless your measurements support that conclusion. The next notebook moves from independent additions to matrix multiplication.

## Profiling: where does the GPU work happen?

In [7]:

import torch

# Warm up both implementations before profiling.
for _ in range(10):
    run_torch()
    run_triton()

torch.cuda.synchronize()

activities = [
    torch.profiler.ProfilerActivity.CPU,
    torch.profiler.ProfilerActivity.CUDA,
]

with torch.profiler.profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
) as prof:
    # Repeated calls make the operations easier to inspect.
    with torch.profiler.record_function("pytorch_vector_add"):
        for _ in range(10):
            run_torch()

    with torch.profiler.record_function("triton_vector_add"):
        for _ in range(10):
            run_triton()

    torch.cuda.synchronize()

print(
    prof.key_averages().table(
        sort_by="self_cuda_time_total",
        row_limit=20,
    )
)

# Save a trace that can be opened in a compatible trace viewer.
prof.export_chrome_trace("/content/triton_lab_profile.json")
print("Saved trace to /content/triton_lab_profile.json")


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                     pytorch_vector_add         0.00%       0.000us         0.00%       0.000us       0.000us     599.855us        57.66%     599.855us     599.855us           0 B           0 B             1  
                                              aten::add        10.01%     689.379us        43.2

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
